# **Raw Data Ingestion**

## **Overview**

This notebook is responsible for ingesting the original Olist e-commerce datasets into the **RAW layer** of the PostgreSQL database.

The RAW layer serves as the initial database landing zone for the source data. Its purpose is to preserve the original datasets in a structured database environment before any further transformation, modelling, or business analysis takes place.

The datasets will be loaded from their original CSV files into PostgreSQL using **Python, Pandas, and SQLAlchemy**. SQLAlchemy provides the connection between the Python environment and the PostgreSQL database, while Pandas is used to read and transfer the datasets.

## **Objectives**

This notebook will:

* Establish a connection between Python and the PostgreSQL `ecommerce` database.
* Load the original Olist datasets into Pandas DataFrames.
* Ingest the datasets into the PostgreSQL `raw` schema.
* Automatically create the corresponding database tables using Pandas and SQLAlchemy.
* Validate the ingestion by comparing source and database record counts.
* Confirm that the ingested data can be successfully queried from PostgreSQL.

## **Data Layer**

The ingestion process follows the architecture established for the project:

**Original Olist datasets → Python/Pandas → SQLAlchemy → PostgreSQL RAW layer**

The RAW layer will contain the source datasets with minimal intervention. Business transformations, analytical structures, relationships, and derived metrics will be handled in later stages of the project.


This notebook focuses specifically on **data ingestion** of raw csv files and does not perform additional data cleaning or analytical modelling.


### **1. Import the Libraries**

In [1]:
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

### **2. Create the SQLAlchemy connection to the Database**

In [2]:
password = getpass("Enter PostgrSQL password:")

In [3]:
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="ecommerce"
)

engine = create_engine(connection_url)

In [4]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print(result.scalar())

1


### **3. Load The Datasets**


In [5]:
customers_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\raw\olist_customers_dataset.csv")
orders_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\raw\olist_orders_dataset.csv")
order_items_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\raw\olist_order_items_dataset.csv")
order_payments_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\raw\olist_order_payments_dataset.csv")
order_reviews_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\raw\olist_order_reviews_dataset.csv")
products_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\raw\olist_products_dataset.csv")
sellers_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\raw\olist_sellers_dataset.csv")
geolocation_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\raw\olist_geolocation_dataset.csv")
translation_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\raw\product_category_name_translation.csv")

#### **Customers Dataset**

In [6]:
# Inspect the dataset
customers_df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [ ]:
# Check the number of rows and columns
customers_df.shape

(99441, 5)

In [ ]:
#Load into the raw schema
customers_df.to_sql(
    name="customers",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False
)

441

In [ ]:
# Validate the ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM raw.customers")
    )
    print(result.scalar())

99441


In [ ]:
# Cross check against the original csv file to check if all the data has been succesfully ingested.
len(customers_df)

99441

In [ ]:
# Check the actual table from Python
pd.read_sql(
    "SELECT * FROM raw.customers LIMIT 5",
    engine
)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


#### **Geolocation dataset**

In [12]:
# Check the number of rows and columns
geolocation_df.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [13]:
# Check the number of rows and columns
geolocation_df.shape

(1000163, 5)

In [17]:
# Load it into the raw schema
geolocation_df.to_sql(
    name="geolocation",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False
)

163

In [ ]:
# validate if the ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM raw.geolocation")
    )
    print(result.scalar())

1000163


In [ ]:
# Check if all of the data from the original csv has been successfully ingested
len(geolocation_df)

1000163

In [22]:
# Check the actual table from python
pd.read_sql(
    "SELECT * FROM raw.geolocation LIMIT 5",
    engine
)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


#### **Order Items Dataset**

In [23]:
# Inspect the dataset
order_items_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [24]:
# Check the number of rows and columns
order_items_df.shape

(112650, 7)

In [28]:
# Load into the raw schema
order_items_df.to_sql(
    name="order_items",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False
)

650

In [31]:
# Validate the Ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM raw.order_items")
    )
    print(result.scalar())

112650


In [32]:
# Check if all of the data from the original csv has been successfully ingested
len(order_items_df)

112650

In [33]:
# Check the actual table from Python
pd.read_sql(
    "SELECT * FROM raw.order_items LIMIT 5",
    engine
)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


#### **Order Reviews Dataset**

In [34]:
# Inspect the datset
order_reviews_df.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [35]:
# Check the number of rows and columns
order_reviews_df.shape

(99224, 7)

In [36]:
# Load into the raw schema
order_reviews_df.to_sql(
    name="reviews",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False
)

224

In [37]:
# Validate the Ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM raw.reviews")
    )
    print(result.scalar())

99224


In [39]:
#  Check if all of the data from the original csv has been successfully ingested
len(order_reviews_df)

99224

In [40]:
# Check the actual table from Python
pd.read_sql(
    "SELECT * FROM raw.reviews LIMIT 5",
    engine
)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


#### **Order Payments Dataset**

In [41]:
# Inspect the dataset
order_payments_df.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [42]:
# Check the number of rows and columns
order_payments_df.shape

(103886, 5)

In [43]:
# Load into the raw schema
order_payments_df.to_sql(
    name="payments",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False
)

886

In [44]:
# Validate the Ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT (*) FROM raw.payments")
    )
    print(result.scalar())

103886


In [45]:
# Check if all of the data from the original csv has been successfully ingested
len(order_payments_df)

103886

In [46]:
# Check the actual table from python
pd.read_sql(
    "SELECT * FROM raw.payments LIMIT 5",
    engine
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


#### **Orders dataset**

In [47]:
# Inspect the dataset
orders_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [48]:
# Check the number of rows and columns
orders_df.shape

(99441, 8)

In [50]:
# Load into the raw schema
orders_df.to_sql(
    name="orders",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False
)

441

In [51]:
# Validate the ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM raw.orders")
    )
    print (result.scalar())

99441


In [52]:
#  Check if all of the data from the original csv has been successfully ingested
len(orders_df)

99441

In [53]:
# Check the actual table from python
pd.read_sql(
    "SELECT * FROM raw.orders LIMIT 5;",
    engine
)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
